In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ── Load Data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('appointments.csv', parse_dates=['appointment_date'])
print("=== Dataset Overview ===")
print(f"Total Records : {len(df)}")
print(f"Date Range    : {df['appointment_date'].min().date()} → {df['appointment_date'].max().date()}")
print(f"Divisions     : {df['division'].nunique()}")
print(f"Specialties   : {df['specialty'].nunique()}\n")

=== Dataset Overview ===
Total Records : 500
Date Range    : 2023-01-02 → 2024-12-26
Divisions     : 8
Specialties   : 6



In [4]:
# ── Basic Cleaning ─────────────────────────────────────────────────────────────
print("=== Missing Values ===")
print(df.isnull().sum())
print()

=== Missing Values ===
patient_age             0
patient_gender          0
division                0
specialty               0
appointment_status      0
consultation_fee_bdt    0
wait_days               0
appointment_date        0
dtype: int64



In [5]:
# Age group segmentation
bins = [0, 17, 35, 55, 100]
labels = ['Child (0-17)', 'Young Adult (18-35)', 'Middle Age (36-55)', 'Senior (56+)']
df['age_group'] = pd.cut(df['patient_age'], bins=bins, labels=labels)

In [6]:
# Extract month name for trend analysis
df['month'] = df['appointment_date'].dt.month
df['year'] = df['appointment_date'].dt.year
df['month_name'] = df['appointment_date'].dt.strftime('%b')

In [7]:
# ── Analysis 1: Appointment Status Breakdown ───────────────────────────────────
status_counts = df['appointment_status'].value_counts()
print("=== Appointment Status ===")
print(status_counts.to_string())
print()

=== Appointment Status ===
appointment_status
Completed    343
Cancelled    101
No-show       56



In [8]:
# ── Analysis 2: No-show Rate by Division ──────────────────────────────────────
noshow_by_division = (
    df.groupby('division')['appointment_status']
    .apply(lambda x: (x == 'No-show').sum() / len(x) * 100)
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
)
noshow_by_division.columns = ['division', 'noshow_rate_pct']
print("=== No-show Rate by Division ===")
print(noshow_by_division.to_string(index=False))
print()

=== No-show Rate by Division ===
  division  noshow_rate_pct
     Dhaka             14.6
Chattogram             13.4
  Rajshahi             11.7
    Sylhet              9.3
Mymensingh              8.0
    Khulna              6.7
  Barishal              4.5
   Rangpur              0.0



In [9]:
# ── Analysis 3: Most In-Demand Specialties ────────────────────────────────────
specialty_demand = df['specialty'].value_counts().reset_index()
specialty_demand.columns = ['specialty', 'appointments']
print("=== Specialty Demand ===")
print(specialty_demand.to_string(index=False))
print()

=== Specialty Demand ===
        specialty  appointments
     Pediatrician            92
    Dermatologist            89
     Cardiologist            85
General Physician            84
     Gynecologist            75
       Orthopedic            75



In [10]:
# ── Analysis 4: Average Wait Days by Specialty ────────────────────────────────
avg_wait = (
    df.groupby('specialty')['wait_days']
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
)
print("=== Average Wait Days by Specialty ===")
print(avg_wait.to_string(index=False))
print()

=== Average Wait Days by Specialty ===
        specialty  wait_days
General Physician       11.6
     Gynecologist       11.5
       Orthopedic       11.5
    Dermatologist       10.5
     Pediatrician        9.3
     Cardiologist        9.0



In [11]:
# ── Analysis 5: Age Group Distribution ───────────────────────────────────────
age_dist = df['age_group'].value_counts().sort_index()
print("=== Patient Age Group Distribution ===")
print(age_dist.to_string())
print()

=== Patient Age Group Distribution ===
age_group
Child (0-17)            90
Young Adult (18-35)    118
Middle Age (36-55)     130
Senior (56+)           162



In [12]:
# ── Analysis 6: Monthly Appointment Trend ─────────────────────────────────────
monthly_trend = (
    df.groupby(['year', 'month', 'month_name'])
    .size()
    .reset_index(name='count')
    .sort_values(['year', 'month'])
)

In [22]:
# ── Key Findings Summary ───────────────────────────────────────────────────────
print("\n=== Key Findings ===")
top_noshow = noshow_by_division.iloc[0]
top_specialty = specialty_demand.iloc[0]
longest_wait = avg_wait.iloc[0]
completion_rate = round(status_counts.get('Completed', 0) / len(df) * 100, 1)

print(f"• Overall completion rate        : {completion_rate}%")
print(f"• Division with highest no-show  : {top_noshow['division']} ({top_noshow['noshow_rate_pct']}%)")
print(f"• Most visited specialty         : {top_specialty['specialty']} ({top_specialty['appointments']} appointments)")
print(f"• Longest average wait           : {longest_wait['specialty']} ({longest_wait['wait_days']} days)")
print(f"• Most common patient group      : {age_dist.idxmax()}")


=== Key Findings ===
• Overall completion rate        : 68.6%
• Division with highest no-show  : Dhaka (14.6%)
• Most visited specialty         : Pediatrician (92 appointments)
• Longest average wait           : General Physician (11.6 days)
• Most common patient group      : Senior (56+)
